# Revision R1: control experiments

New inference for the reviewer response. Run on a Colab **L4 or A100** GPU, as for
the original study. Nothing here changes the responses cached for the submitted
paper: new prompts go to new cache files (`P4`-`P6`) and new pair sets to new
folders.

| Stage | Pair sets | Prompts | Answers which comment |
|---|---|---|---|
| A | the paper's four sets | P4 persona only, P5 length-matched neutral, P6 persona + length, no fairness wording | prompt packages confound persona, length and fairness wording |
| B | placebo (right-/left-handed), sex pairs with both versions edited, reverse-direction sex pairs (female original) | P1-P3 | original text versus edited text; wording change alone |
| C | every pair the rules can build (9,699 / 4,876 / 8,226 / 4,152) | P1-P3 | sample size |
| D (optional) | placebo for every race-eligible vignette | P1-P3 | placebo at the same precision as stage C |

Token-probability confidence only, which is the primary analysis; set `RUN_CE = True`
to add elicited confidence for stages A and B. Every response is cached as it is
produced, so a disconnected session resumes where it stopped.

## 0. Setup

Same Drive layout as the original notebook (`My Drive/CARE/{source,pairs,results,outputs}`).
The code is taken from the **`r1-revision`** branch; push that branch to GitHub first,
or upload the repository folder to Colab and set `CODE` to it.

In [ ]:
!pip -q install "transformers>=5.8" accelerate statsmodels
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO_URL = "https://github.com/haohuilu/CARE.git"
BRANCH = "r1-revision"
CODE = "/content/CARE"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, CODE], check=True)
COMMIT = subprocess.run(["git", "-C", CODE, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", BRANCH, COMMIT[:12])

DRIVE = "/content/drive/MyDrive/CARE"
os.environ["CARE_SOURCE_DIR"] = f"{DRIVE}/source"
os.environ["CARE_PAIRS_DIR"] = f"{DRIVE}/pairs"
os.environ["CARE_DATA_ROOT"] = f"{DRIVE}/results"
os.environ["CARE_OUTPUT_ROOT"] = f"{DRIVE}/outputs"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN"); print("  OK       HF_TOKEN")
except Exception:
    print("  MISSING  HF_TOKEN -- key icon on the left > Add new secret, and enable notebook access")
os.chdir(CODE)

In [ ]:
import sys, json, time, gc
from pathlib import Path
sys.path.insert(0, CODE)
import pandas as pd, torch
from care import config as cfg, infer, controls, revision
from care.prompts import SYSTEM, CONDITIONS, CONTROL_CONDITIONS

RESULTS = Path(cfg.DATA_ROOT)
DEVICE, DTYPE = infer.pick_device()
print("device:", DEVICE, DTYPE, "| results ->", RESULTS)

# What to run. Stage C is the long one; A and B answer most of the comments.
RUN_FULL = True            # stage C: every available pair
RUN_PLACEBO_FULL = False   # stage D
RUN_CE = False             # elicited confidence for stages A and B

## 1. Pair sets

Built by rule from the source corpora, like the paper's pairs. The four pair files of
the paper must already be in `pairs/`; their hashes are checked and they are never
rewritten. The hashes printed below should match those in the repository README.

In [ ]:
# Pair files: use the ones on Drive if they are all there (checked by hash);
# build them from the source corpora only if some are missing.
import hashlib
EXPECTED = {
    "pairs_medqa_race": "d32c84414db706f4", "pairs_medqa_gender": "084ad9446b00e11d",
    "pairs_medmcqa_race": "b2058037d8772b85", "pairs_medmcqa_gender": "e12d901caa9ff4a7",
    "pairs_medqa_placebo": "0697c458768d84cc", "pairs_medqa_gender_edited": "d6a3d8ea99e4ad56",
    "pairs_medqa_gender_reverse": "bb0532e911e26fa3", "pairs_medqa_race_full": "6e93930dd192d13f",
    "pairs_medqa_gender_full": "ee692c1d53a602cd", "pairs_medqa_placebo_full": "82be41fd725ff9bf",
    "pairs_medmcqa_placebo": "0b3b9b89f8edd19a", "pairs_medmcqa_gender_edited": "bc424f5791c77b51",
    "pairs_medmcqa_gender_reverse": "a0a40b986353d884", "pairs_medmcqa_race_full": "098a2567be174910",
    "pairs_medmcqa_gender_full": "7714d0a567d65d09", "pairs_medmcqa_placebo_full": "850d5671a5e5b1be",
}
PAIRS_DIR, SOURCE_DIR = Path(cfg.PAIRS_DIR), Path(cfg.SOURCE_DIR)
kinds = {"placebo", "both_edited", "reverse"} | ({"full"} if RUN_FULL else set()) \
        | ({"placebo_full"} if RUN_PLACEBO_FULL else set())
NEEDED = list(cfg.MAIN_DATASETS) + [d for d in cfg.CONTROL_DATASETS if cfg.DATASETS[d]["kind"] in kinds]

def sha16(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

missing = [d for d in NEEDED if not (PAIRS_DIR / f"{d}.jsonl").exists()]
if missing:
    print(f"{len(missing)} pair file(s) not in {PAIRS_DIR}: {missing}")
    have_src = (SOURCE_DIR / "MedQA" / "train.json").exists() and (SOURCE_DIR / "medmcqa" / "train.json").exists()
    if not have_src:
        raise SystemExit(
            "Cannot build them: the source corpora are not on Drive either.\n"
            f"  looked for {SOURCE_DIR}/MedQA/train.json and {SOURCE_DIR}/medmcqa/train.json\n"
            "EASIEST FIX: upload the folder  care-referral-fairness/data/pairs  from your Mac\n"
            f"so that its files sit directly in  {PAIRS_DIR}/  , then run this cell again.")
    if any(d in missing for d in cfg.MAIN_DATASETS):
        r = subprocess.run([sys.executable, "-m", "care.pairgen", "--outdir", str(PAIRS_DIR)],
                           capture_output=True, text=True)
        print(r.stdout[-2000:], r.stderr[-3000:])
        r.check_returncode()
    controls.build_all(str(PAIRS_DIR), placebo_full=RUN_PLACEBO_FULL, full=RUN_FULL)

bad = {d: sha16(PAIRS_DIR / f"{d}.jsonl") for d in NEEDED
       if sha16(PAIRS_DIR / f"{d}.jsonl") != EXPECTED[d]}
if bad:
    raise SystemExit(f"These pair files differ from the ones built for the revision: {bad}")
print(f"{len(NEEDED)} pair files present, all hashes as expected")
if not (PAIRS_DIR / "pair_flags.csv").exists():
    print("NOTE: pair_flags.csv is not in the pairs folder; upload it too "
          "(the split and sensitivity analyses need it)")

def load_pairs(name):
    return [json.loads(l) for l in open(PAIRS_DIR / f"{name}.jsonl", encoding="utf-8") if l.strip()]

N_OPTIONS = {d: s["n_options"] for d, s in cfg.DATASETS.items()}
MAIN = {d: load_pairs(d) for d in cfg.MAIN_DATASETS}
CTRL = {d: load_pairs(d) for d in cfg.CONTROL_DATASETS
        if cfg.DATASETS[d]["kind"] in ("placebo", "both_edited", "reverse")}
FULL = {d: load_pairs(d) for d in cfg.CONTROL_DATASETS
        if (cfg.DATASETS[d]["kind"] == "full" and RUN_FULL)
        or (cfg.DATASETS[d]["kind"] == "placebo_full" and RUN_PLACEBO_FULL)}
for grp, name in ((MAIN, "paper"), (CTRL, "control"), (FULL, "full")):
    for d, r in grp.items():
        print(f"{name:8s} {d:32s} {len(r)//2:6d} pairs  {sorted({x['demographic'] for x in r})}")

# The responses cached for the paper must be on Drive too: the full-pool run
# starts from them and the analysis compares the new runs with them.
MODEL_DIRS = ["google_medgemma-4b-it", "google_gemma-4-E4B-it",
              "m42-health_Llama3-Med42-8B", "meta-llama_Llama-3.1-8B-Instruct"]
gone = [f"{d}/{m}/{c}" for d in cfg.MAIN_DATASETS for m in MODEL_DIRS for c in ("P1", "P2", "P3")
        if not (RESULTS / d / m / "calibration_cache" / f"{c}.jsonl").exists()]
print("paper's cached responses: all 48 files found" if not gone else
      f"WARNING: {len(gone)} of 48 cached response files of the paper are missing under {RESULTS}\n"
      f"  e.g. {gone[:3]}\n  Upload the folders inside  CARE/final/results  from your Mac into  {RESULTS}/  "
      "before running section 4.")

Read some pairs before spending compute, the reverse-direction set in particular: there
the female version is the benchmark text and the male version is produced by rule.
`pairs/review_<set>.md` on Drive lists 40 sampled pairs per set with the changed words in bold.

In [ ]:
import difflib, random
def show_pair(dataset, i=None, width=100):
    recs = {**MAIN, **CTRL, **FULL}[dataset]
    ids = sorted({r["pair_id"] for r in recs})
    pid = ids[i] if i is not None else random.choice(ids)
    a, b = [r for r in recs if r["pair_id"] == pid]
    ta, tb = a["question"].split(), b["question"].split()
    ma, mb = [], []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, ta, tb).get_opcodes():
        ma += ta[i1:i2] if op == "equal" else [f"[{w}]" for w in ta[i1:i2]]
        mb += tb[j1:j2] if op == "equal" else [f"[{w}]" for w in tb[j1:j2]]
    print(f"=== {dataset} | pair {pid} ===")
    for rec, words in ((a, ma), (b, mb)):
        print(f"[{rec['demographic'].upper()}] " + " ".join(words)[:900])
    print()
for d in CTRL:
    show_pair(d, i=0)

## 2. Prompts

`P4`-`P6` carry no fairness wording. `P4 - P1` is the persona, `P5 - P1` the added length
and clinical framing, and `P2 - P6`, `P3 - P6` the fairness guidance with persona and
length held fixed.

In [ ]:
for c in CONDITIONS + CONTROL_CONDITIONS:
    print(f"--- {c} ({len(SYSTEM[c].split())} words) ---\n{SYSTEM[c]}\n")

## 3. Plan and time estimate

In [ ]:
LOCAL = {                                   # HF id -> results directory (as in the paper)
    "google/medgemma-4b-it":            "google_medgemma-4b-it",
    "google/gemma-4-E4B-it":            "google_gemma-4-E4B-it",
    "m42-health/Llama3-Med42-8B":       "m42-health_Llama3-Med42-8B",
    "meta-llama/Llama-3.1-8B-Instruct": "meta-llama_Llama-3.1-8B-Instruct",
}
STAGES = [("A control prompts", MAIN, CONTROL_CONDITIONS),
          ("B control pairs",   CTRL, CONDITIONS),
          ("C/D full pools",    FULL, CONDITIONS)]
STAGES = [s for s in STAGES if s[1]]

# Stage C starts from the responses already cached for the 1,000-pair sets,
# so only the pairs that are new are computed.
seeded = controls.seed_full_cache(RESULTS, list(LOCAL.values()), CONDITIONS)
print(f"seeded {len(seeded)} full-pool cache files from the existing responses")

def todo_count(mdir):
    n = 0
    for _, pairs, conds in STAGES:
        for ds, recs in pairs.items():
            for c in conds:
                seen = infer.done_keys(infer.out_path(RESULTS, ds, mdir, c), "tlp")
                n += sum((str(r["question_id"]), r["variant"]) not in seen for r in recs)
    return n
TODO = {name: todo_count(mdir) for name, mdir in LOCAL.items()}
for name, n in TODO.items():
    print(f"{name:36s} {n:8,d} forward passes to run")

BATCH, DEFAULT_BATCH, META = {}, (16 if DEVICE == "cuda" else 8), {}
def free(model=None):
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def smoke(name, n=16):
    tok, model = infer.load_local(name, DEVICE, DTYPE)
    ids = infer.letter_token_ids(tok)
    recs = MAIN["pairs_medqa_race"][:n]
    texts = [infer.chat_text(tok, SYSTEM["P6"], infer.user_prompt(r)) for r in recs]
    t0 = time.time(); batched = infer.tlp_batch(model, tok, ids, texts, infer.LETTERS[:5]); dt = (time.time() - t0) / n
    single = [infer.tlp_batch(model, tok, ids, [t], infer.LETTERS[:5])[0] for t in texts[:4]]
    diff = max(abs(a["probs"][k] - b["probs"][k]) for a, b in zip(single, batched) for k in a["probs"])
    mass = sorted(r["letter_mass"] for r in batched)[n // 2]
    print(f"{name}\n  median letter mass {mass:.2f} | batch-vs-single diff {diff:.1e} | "
          f"{dt:.3f}s/item -> ~{TODO[name] * dt / 3600:.1f} h for everything still to run")
    if mass < 0.5: print("  WARNING: low letter mass under P6 -- inspect before running")
    if diff > 0.02: BATCH[name] = 1; print("  batch size 1 for this model")
    free(model)
for name in LOCAL:
    try: smoke(name)
    except Exception as e: print(f"{name}\n  FAILED: {type(e).__name__}: {str(e)[:300]}"); free()

## 4. Run

One model at a time, stages in order A, B, C. Re-run the cell after a disconnect; it
resumes. To run only A and B first, set `RUN_FULL = False` above and re-run from section 1.

In [ ]:
for name, mdir in LOCAL.items():
    print(f"\n=== {name} ===")
    try:
        tok, model = infer.load_local(name, DEVICE, DTYPE)
        META[name] = {"backend": "local", "dir": mdir, "dtype": str(DTYPE),
                      "revision": getattr(model.config, "_commit_hash", None)}
        bs = BATCH.get(name, DEFAULT_BATCH)
        for label, pairs, conds in STAGES:
            print(f"-- stage {label}")
            infer.run_local_tlp(model, tok, name, mdir, pairs, N_OPTIONS, SYSTEM, conds, RESULTS, batch_size=bs)
            if RUN_CE and not label.startswith("C"):
                infer.run_local_ce(model, tok, name, mdir, pairs, N_OPTIONS, SYSTEM, conds, RESULTS, batch_size=bs)
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {str(e)[:300]} -- re-run this cell to resume")
    finally:
        model = None; free()

## 5. Completeness and manifest

The checkpoint revisions should equal those in Table B1 of the paper; if a model was
updated on Hugging Face since September, the new runs are not comparable with the cached
ones and that model has to be pinned to the old revision.

In [ ]:
from care.data import load_records
rows = []
for label, pairs, conds in STAGES:
    for ds, recs in pairs.items():
        for mdir in LOCAL.values():
            try:
                df = load_records(ds, mdir, "tlp", conditions=list(conds))
                n = len(df)
            except FileNotFoundError:
                n = 0
            rows.append({"stage": label[0], "dataset": ds, "model": cfg.short(mdir),
                         "n": n, "expected": len(recs) * len(conds)})
chk = pd.DataFrame(rows); chk["complete"] = chk.n >= chk.expected
print(chk.to_string(index=False))
print("ALL COMPLETE" if chk.complete.all() else "INCOMPLETE -- re-run section 4")

PAPER_REVISIONS = {"google/medgemma-4b-it": "290cda5eeccbee130f987c4ad74a59ae6f196408",
                   "m42-health/Llama3-Med42-8B": "ceab7e7ee4b9dbde7ba82867f34274db51487d83",
                   "google/gemma-4-E4B-it": "ee0ef6023621cff504d758262d4e04895a5af4a2",
                   "meta-llama/Llama-3.1-8B-Instruct": "0e9e39f249a16976918f6564b8830bc894c89659"}
for name, m in META.items():
    same = m.get("revision") == PAPER_REVISIONS.get(name)
    print(("  same revision as the paper   " if same else "  DIFFERENT REVISION           "), name, m.get("revision"))

files = {d: str(Path(cfg.PAIRS_DIR) / f"{d}.jsonl") for grp in (MAIN, CTRL, FULL) for d in grp}
print(infer.write_manifest(Path(cfg.RESULTS_DIR) / "r1_run_manifest.json", files, SYSTEM, META,
                           extra={"code_branch": BRANCH, "code_commit": COMMIT}))

## 6. Analysis

Writes `outputs/csv/r1_*.csv` and `r1_summary.md`. The first call covers the paper's
1,000-pair sets and the control experiments; the second repeats the main analyses on
every available pair.

In [ ]:
revision.main(["--stage", "all"])
if RUN_FULL:
    revision.main(["--stage", "cache", "--tag", "full", "--datasets",
                   *[d for d in FULL if cfg.DATASETS[d]["kind"] == "full"]])
print(open(Path(cfg.RESULTS_DIR) / "r1_summary.md").read())

In [ ]:
show = lambda name, cols=None, **kw: (lambda d: print(f"\n=== {name} ===\n" + (d[cols] if cols else d).to_string(index=False, float_format=lambda v: f"{v:+.4f}")))(pd.read_csv(Path(cfg.RESULTS_DIR) / f"r1_{name}.csv"))
show("controls", ["parent", "kind", "model", "prompt", "n_pairs", "d_conf", "d_conf_lo", "d_conf_hi",
                  "answer_changed", "sar_gap", "sar_gap_lo", "sar_gap_hi", "auroc_diff"])
show("edit_effect")
show("prompt_controls", ["dataset", "model", "contrast", "isolates", "d_gap", "d_gap_lo", "d_gap_hi",
                         "d_sar_mean", "d_sar_mean_lo", "d_sar_mean_hi", "d_acc_mean"])